# CH08 - Laboratorio: Árboles Binarios

Laboratorio del capítulo 8 (Goodrich, Tamassia & Goldwasser).

A diferencia del laboratorio de listas enlazadas (CH07), aquí **no reimplementamos el ADT** desde cero: usamos directamente `LinkedBinaryTree` del repositorio del curso (`goodrich.ch08.linked_binary_tree`). El laboratorio se concentra en resolver problemas *sobre* un árbol binario ya funcional.

Completa **todas** las funciones marcadas con `raise NotImplementedError`.
Cada ejercicio trae sus propias pruebas: ejecútalas para verificar tu solución.

**Contenido**

0. Implementación base - `LinkedBinaryTree` (Goodrich) y árbol de prueba
1. Ejercicio - Contar los primos de un nodo
2. Ejercicio - ¿Son primos dos nodos?
3. Ejercicio (C-8.58) - Primer ancestro común (LCA)
4. Ejercicio - La sucesión al poder en el reino de los números


---
## 0. Implementación base - `LinkedBinaryTree` (Goodrich)

Usamos la implementación del árbol binario del repositorio del curso. Ya trae resuelto todo el ADT:

| Método | Descripción |
|---|---|
| `root()`, `parent(p)`, `left(p)`, `right(p)` | Navegación básica |
| `sibling(p)` | Hermano de `p` (mismo padre) |
| `children(p)`, `num_children(p)` | Hijos de `p` |
| `is_root(p)`, `is_leaf(p)`, `is_empty()` | Consultas booleanas |
| `depth(p)`, `height(p)` | Profundidad y altura |
| `preorder()`, `inorder()`, `postorder()`, `breadthfirst()`, `positions()` | Recorridos completos |

**Diferencia importante con `ch08_teoria`:** aquí cada posición es un objeto `Position`, no un nodo crudo. Se compara con `==` (no con `is`) y el elemento se obtiene con `p.element()` (no `p._element`).

Para construir árboles de prueba usamos directamente los métodos de mutación `_add_root(e)`, `_add_left(p, e)`, `_add_right(p, e)` (llevan guion bajo porque en el diseño de Goodrich el ADT es de solo lectura desde afuera; para este laboratorio basta con usarlos así).

El árbol de prueba que se usa en los dos ejercicios es:

```
          A                     nivel 0
        /   \
       B     C                   nivel 1
      / \   / \
     D   E F   G                  nivel 2
        /       \
       H         I                 nivel 3
```

`H` e `I` están al **mismo nivel** (3), pero cuelgan de ramas distintas: el abuelo de `H` es `B` y el de `I` es `C`. Este par se usa a propósito en los ejercicios siguientes para distinguir "mismo nivel" de "primos".</cell id="766cb550">


In [ ]:
from goodrich.ch08.linked_binary_tree import LinkedBinaryTree


def construir_arbol_primos():
    """Construye el árbol de prueba (A-I) usado en los ejercicios de esta sección."""
    T = LinkedBinaryTree()
    a = T._add_root('A')
    b = T._add_left(a, 'B')
    c = T._add_right(a, 'C')
    d = T._add_left(b, 'D')
    e = T._add_right(b, 'E')
    f = T._add_left(c, 'F')
    g = T._add_right(c, 'G')
    h = T._add_left(e, 'H')
    i = T._add_right(g, 'I')
    return T, dict(A=a, B=b, C=c, D=d, E=e, F=f, G=g, H=h, I=i)


# --- Prueba rápida ---
T, N = construir_arbol_primos()
print(len(T))                                       # 9
print(T.root().element())                           # A
print(T.depth(N['H']), T.depth(N['I']))              # 3 3  (mismo nivel)
print(T.sibling(N['D']).element())                   # E
print([c.element() for c in T.children(N['C'])])     # ['F', 'G']


---
## Ejercicio 1 - Contar los primos de un nodo

Igual que en un árbol genealógico, dos nodos de un árbol binario son **primos** si:

1. Tienen el **mismo abuelo** (`T.parent(T.parent(·))`), y
2. Tienen **padres distintos** — si compartieran padre serían **hermanos** (`sibling`), no primos.

**Ojo:** "mismo nivel" (`depth`) **no alcanza** — es necesario, pero no suficiente. Sobre el árbol de la sección 0, `H` e `I` están al mismo nivel (3), pero el abuelo de `H` es `B` y el de `I` es `C`: no son primos.

Sobre el mismo árbol:

- `D` y `E` son **hermanos** (mismo padre `B`) → no son primos entre sí.
- `D` y `F` sí son primos (mismo abuelo `A`, padres `B` y `C`, que son distintos).
- `H` no tiene primos: el único otro nieto de `B` sería un hermano de `E`, y no existe.

Implementa `contar_primos(T, p)`: recibe el árbol `T` y una posición `p`, y retorna cuántos primos tiene `p`.

**Pista:** calcula el abuelo de `p` con `T.parent(T.parent(p))` (cuidado: si `p` es la raíz o un hijo de la raíz, no tiene abuelo — retorna `0` en ese caso). Luego recorre `T.positions()`, filtra las que tengan el mismo abuelo que `p`, y descarta las que además tengan el mismo padre que `p` (esas son hermanas de `p`, no primas).

In [ ]:
def contar_primos(T, p):
    """Retorna el número de primos (mismo abuelo, padre distinto) de la posición p."""
    raise NotImplementedError


# --- Tests ---
T, N = construir_arbol_primos()

print(contar_primos(T, N['D']))   # 2  (F y G)
print(contar_primos(T, N['E']))   # 2  (F y G)
print(contar_primos(T, N['F']))   # 2  (D y E)
print(contar_primos(T, N['H']))   # 0  (nadie más comparte abuelo B)
print(contar_primos(T, N['I']))   # 0  (nadie más comparte abuelo C)
print(contar_primos(T, N['A']))   # 0  (la raíz no tiene abuelo)
print(contar_primos(T, N['B']))   # 0  (hijo de la raíz, no tiene abuelo)


---
## Ejercicio 2 - ¿Son primos dos nodos?

Ahora, dado un árbol y **dos posiciones** `p` y `q`, determina si son primos entre sí (misma definición del ejercicio 1: mismo abuelo, padres distintos).

Implementa `son_primos(T, p, q)`. Debe retornar `False` si `p` y `q` son la misma posición, y también si alguno de los dos no tiene abuelo (es la raíz o un hijo de la raíz).

In [ ]:
def son_primos(T, p, q):
    """Retorna True si p y q son primos: mismo abuelo y padres distintos."""
    raise NotImplementedError


# --- Tests ---
T, N = construir_arbol_primos()

print(son_primos(T, N['D'], N['F']))   # True
print(son_primos(T, N['D'], N['E']))   # False (hermanos)
print(son_primos(T, N['D'], N['D']))   # False (mismo nodo)
print(son_primos(T, N['A'], N['B']))   # False (A no tiene abuelo)
print(son_primos(T, N['H'], N['I']))   # False (mismo nivel, pero abuelos distintos: B y C)
print(son_primos(T, N['F'], N['G']))   # False (hermanos)


---
## Ejercicio 3 (C-8.58) - Primer ancestro común (LCA)

> *Let T be a tree with n positions. Define the lowest common ancestor (LCA) between two positions p and q as the lowest position in T that has both p and q as descendants (where we allow a position to be a descendant of itself). Given two positions p and q, describe an efficient algorithm for finding the LCA of p and q.*

El **primer ancestro común** (LCA) de dos posiciones `p` y `q` es la posición **más profunda** (más "abajo", más lejos de la raíz) que es ancestro de ambas. Por convención, una posición es ancestro de sí misma, así que `LCA(p, p) = p`.

**Idea del algoritmo eficiente (sin construir el conjunto completo de ancestros):**

1. Calcula `depth(p)` y `depth(q)`.
2. **Nivela** las dos posiciones: mientras una esté más profunda que la otra, súbela con `T.parent(...)` hasta que ambas queden a la misma profundidad. Esto no puede saltarse el LCA, porque el LCA está a una profundidad menor o igual que la más superficial de `p` y `q`.
3. Ahora que `p` y `q` están al mismo nivel, sube **ambas a la vez**, un paso por llamada (`p = T.parent(p)`, `q = T.parent(q)`), hasta que sean la **misma posición** (`p == q`). Esa posición es el LCA.

**Por qué es eficiente:** cada paso sube un nivel por alguna de las dos posiciones, así que el algoritmo hace a lo más $O(\text{depth}(p) + \text{depth}(q)) = O(h)$ llamadas a `parent`, donde $h$ es la altura del árbol — mucho mejor que $O(n)$ (comparar contra todas las posiciones).

Sobre el árbol de la sección 0:

- `LCA(D, E) = B` (mismo padre).
- `LCA(D, F) = A` (cuelgan de subárboles distintos de la raíz).
- `LCA(D, H) = B`: los ancestros de `D` son `{D, B, A}`; los de `H` son `{H, E, B, A}`; el más profundo en común es `B`.
- `LCA(H, I) = A`: aunque están al mismo nivel, sus ramas solo se juntan en la raíz.
- `LCA(D, D) = D` (una posición es ancestro de sí misma).

Implementa `lca(T, p, q)` siguiendo el algoritmo de nivelar y subir en paralelo.

In [ ]:
def lca(T, p, q):
    """Retorna la posición del primer ancestro común (LCA) de p y q."""
    raise NotImplementedError


# --- Tests ---
T, N = construir_arbol_primos()

print(lca(T, N['D'], N['E']).element())   # B
print(lca(T, N['D'], N['F']).element())   # A
print(lca(T, N['D'], N['H']).element())   # B
print(lca(T, N['H'], N['I']).element())   # A
print(lca(T, N['D'], N['D']).element())   # D
print(lca(T, N['A'], N['I']).element())   # A


---
## Ejercicio 4 - La sucesión al poder en el reino de los números

En el **reino de los números** el poder se hereda siguiendo un árbol binario de descendencia: la raíz es quien ostenta el poder y cada nodo es un descendiente. Por culpa de las guerras internas, la línea de sucesión tiene dos reglas:

1. **Orden de búsqueda.** El primero en heredar es el hijo que está a la **derecha** del que tiene el poder. Si en esa rama no queda nadie apto, se salta a la **descendencia del hijo de la izquierda**. La regla se aplica igual dentro de cada rama: derecha primero, izquierda después.
2. **Traidores.** Un número **no puede** obtener el poder si la **suma de sus dígitos es par** o si **termina en 3** (los de esa categoría suelen ser traicioneros). Un traidor queda excluido del trono, pero **su descendencia sigue en la línea de sucesión**: se lo salta a él, no a su rama.

Escribe un **generador** `herederos(T, p=None)` que produzca, una a una y en orden, las posiciones de la línea de poder a partir de `p` (por defecto, la raíz). La línea **encabeza con `p`**, que es quien ostenta el poder, y sigue con sus sucesivos herederos. La regla de los traidores se aplica a todos por igual, `p` incluido.

En este ejercicio **no se entregan los encabezados**: tú escribes desde cero, en la celda vacía de abajo, la función `es_traidor(n)` (retorna `True` si `n` es traidor) y el generador `herederos(T, p=None)`. Respeta esos nombres y firmas: la celda de pruebas los usa.

Árbol del reino usado en las pruebas (entre paréntesis, los traidores):

```
                    41  (rey)
              /                \
            27                  84 (traidor: 8+4=12 par)
         /      \            /       \
       16     23 (term. 3)  50        63 (termina en 3)
      /  \     /   \       /  \      /
    30    12  70    61   10    32   45
```

Recorrido esperado: encabeza `41`, que ostenta el poder. Luego se entra al subárbol de `84`; como `84` es traidor se sigue con su rama derecha (`63`, también traidor) y de ahí sale el primer heredero, `45`. Recién agotada toda la rama derecha se pasa a la descendencia de `27`.

**Pistas**

- Suma de dígitos de `n`: `sum(int(d) for d in str(abs(n)))`. Termina en 3: `abs(n) % 10 == 3`.
- Es un recorrido en **preorden invertido** (visita el nodo, luego la derecha, luego la izquierda) que *filtra* al producir, pero **no poda**: al traidor no se lo produce, aunque sí se baja a sus hijos.
- Un generador recursivo se escribe con `yield` para el nodo actual y `yield from` para las subllamadas — igual que `_subtree_preorder` en `goodrich/ch08/tree.py`.
- Como es un generador, es **perezoso**: `next(g)` debe entregar el siguiente heredero sin recorrer el árbol completo.


In [ ]:
# Escribe aquí es_traidor(n) y el generador herederos(T, p=None)


In [ ]:
# --- Árbol del reino ---
R = LinkedBinaryTree()
a = R._add_root(41)
b = R._add_left(a, 27);  c = R._add_right(a, 84)
d = R._add_left(b, 16);  e = R._add_right(b, 23)
f = R._add_left(c, 50);  g = R._add_right(c, 63)
R._add_left(d, 30);      R._add_right(d, 12)
R._add_left(e, 70);      R._add_right(e, 61)
R._add_left(f, 10);      R._add_right(f, 32)
R._add_left(g, 45)

# --- Tests ---
print([es_traidor(n) for n in (84, 63, 23, 41, 45, 30)])
# [True, True, True, False, False, False]

print([p.element() for p in herederos(R)])
# [41, 45, 50, 32, 10, 27, 61, 70, 16, 12, 30]

linea = herederos(R)                  # el generador es perezoso
print(next(linea).element(), next(linea).element())   # 41 45

print([p.element() for p in herederos(R, R.left(R.root()))])    # línea que encabeza 27
# [27, 61, 70, 16, 12, 30]

print([p.element() for p in herederos(R, R.right(R.root()))])   # 84 es traidor: no encabeza
# [45, 50, 32, 10]

print([p.element() for p in herederos(R, R.left(R.left(R.root())))])   # línea que encabeza 16
# [16, 12, 30]

print(list(herederos(LinkedBinaryTree())))   # []  (reino sin nadie)
